## Praktikum 2
##### Erweiterte Operationen auf Gesichtern

In diesem Praktikum betrachten wir weitere Operationen auf Gesichtern und trainieren erste kleine Netzwerke zur Gesichtserkennung.

[OpenFace](http://reports-archive.adm.cs.cmu.edu/anon/2016/CMU-CS-16-118.pdf)

[OpenCVs DNN-Sample für OpenFace](https://github.com/opencv/opencv/blob/master/samples/dnn/js_face_recognition.html)

[OpenCVs FaceRec-Tutorial](https://docs.opencv.org/3.3.1/da/d60/tutorial_face_main.html)

[OpenCVs FaceRec-Dokumentation](https://docs.opencv.org/3.3.1/db/d7c/group__face.html)

<div style="padding: 10px; border-left: 5px solid #B777FA; background-color: #5C18FA; border-radius: 4px">
  🐍 Bitte nutzen Sie die virtuelle Umgebung: <b>DLACV_2526</b>
</div>

In [2]:
# imports
import os, sys, dlib
import cv2
import math
import numpy as np
import utils.faceBlendCommon as fbc
import matplotlib.pyplot as plt
import matplotlib
import _pickle as cPickle
import utils.faceUtils as faceUtils
from utils.dataPath import DATA_PATH
%matplotlib inline

In [3]:
# matplotlib-Einstellungen
matplotlib.rcParams['figure.figsize'] = (6.0,6.0)
matplotlib.rcParams['image.cmap'] = 'gray'
matplotlib.rcParams['image.interpolation'] = 'bilinear'

# Detektoren initialisieren
faceDetector = dlib.get_frontal_face_detector() 
landmarkDetector = dlib.shape_predictor(DATA_PATH + "models/shape_predictor_68_face_landmarks.dat")

# Konstanten
faceWidth = 64
faceHeight = 64

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.1:</b><br>
  • Laden Sie die Bilder aus dem letzten Praktikum <a href="https://fh-aachen.sciebo.de/s/mQM6wExJoxeBjQz">hier</a> herunter und entpacken es in den Ordner <code>data/images/custom_faces</code>.
</div>
<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #66611F; 
    border-radius: 4px; 
    margin-top: 5px;
">
💡 <i><b>Hinweis:</b></i> Je nachdem wann sie den Link aufrufen, sind die Daten womöglich noch nicht verfügbar. Bitte warten Sie in diesem Fall, bis eine entsprechende Nachricht in die Webex-Gruppe geschickt wird.
</div>

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.2:</b><br>
  • Detektieren Sie die Landmarken auf den Gesichtsbildern.
  • Richten Sie die Gesichter auf die Werte <code>faceWidth</code> und <code>faceHeight</code> aus.
</div>
<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #66611F; 
    border-radius: 4px; 
    margin-top: 5px;
">
💡 <i><b>Hinweis 1:</b></i> Womöglich brauchen Sie eine oder mehrere Funktionen aus <code>fbc</code>.<br>
💡 <i><b>Hinweis 2:</b></i> Dieser Code kann abhängig von der Hardware mehrere Minuten brauchen, um durchzulaufen.
</div>

In [4]:
# Trainingsdaten vorbereiten
imagesFolder = DATA_PATH + 'images/custom_faces'
subfolders = []

for x in os.listdir(imagesFolder):
    xpath = os.path.join(imagesFolder, x)
    if os.path.isdir(xpath):
        subfolders.append(xpath)

In [5]:
# Bilder labeln
imagePaths = []
labels = []

labelsMap = {}
labelsMap[-1] = "unknown"

for i, subfolder in enumerate(subfolders):
    labelsMap[i] = os.path.basename(subfolder)
    for x in os.listdir(subfolder):
        xpath = os.path.join(subfolder, x)
        if x.lower().endswith(('.jpg', '.jpeg', '.png', '.pgm')):
            imagePaths.append(xpath)
            labels.append(i)

imagesFaceTrain = []
labelsFaceTrain = []

In [ ]:
# Gesichter ausrichten
for j, imagePath in enumerate(imagePaths):
    im = cv2.imread(imagePath, 0)
    
    if im is None:
        print(f"Warnung: Bild konnte nicht geladen werden: {imagePath}")
        continue

    imHeight, imWidth = im.shape[:2]
    
    # Landmarken detektieren

    landmarks = fbc.getLandmarks(faceDetector,landmarkDetector, im) 
    if len(landmarks) < 68:
        continue

    # Ränder berechnen
    x1Limit = landmarks[0][0] - (landmarks[36][0] - landmarks[0][0])
    x2Limit = landmarks[16][0] + (landmarks[16][0] - landmarks[45][0])
    y1Limit = landmarks[27][1] - 3*(landmarks[30][1] - landmarks[27][1])
    y2Limit = landmarks[8][1] + (landmarks[30][1] - landmarks[29][1])
    imFace = im[y1Limit:y2Limit,x1Limit:x2Limit]
    if imFace.shape[0] > 0 and imFace.shape[1] > 0 :
    # Gesicht ausrichten
        alignedFace = faceUtils.alignFace(imFace, landmarks)
        alignedFace = cv2.resize(alignedFace, (faceHeight, faceWidth))

        # Ergebnis sichern
        imagesFaceTrain.append(np.float32(alignedFace)/255.0)
        labelsFaceTrain.append(labels[j])

Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Katie McGrath\030.png
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Katie McGrath\060.png
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Katie McGrath\073.png
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Katie McGrath\081.png
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\5DRZ5SMI4NCAPDS6BEBNKITUSU.jpg
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\dehwkwehrg.jpg
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\gdsufzguegir.jpg
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\mads-mikkelsen1ewrwr4.jpg
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\mads-WZAVEEH7EVE57GH3PADXT64TRU.jpg
Warnung: Bild konnte nicht geladen werden: data/images/custom_faces\Mads Mikkelsen\mh-7-2-mads-15937

<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #3498; 
    border-radius: 4px;
">
  📌 <b>Aufgabe 3.3:</b><br>
  Erstellen, trainieren und serialisieren Sie <code>cv2</code>-Face-Recognizer basierend auf:<br>
  • Eigen-Faces.<br>
  • Fisher-Faces.<br>
  • LBP-Histogrammen.
</div>
<div style="
    padding: 12px; 
    border-left: 5px solid #e8f4fc; 
    background-color: #66611F; 
    border-radius: 4px; 
    margin-top: 5px;
">
💡 <i><b>Hinweis 1:</b></i> Dieser Code kann abhängig von der Hardware mehrere Minuten brauchen, um durchzulaufen.<br>
💡 <i><b>Hinweis 2:</b></i> Achten Sie darauf, die Dateipfade zum Abspeichern nicht zu verändern.
</div>

In [ ]:
print("Trainiere mit Eigen-Faces")
save_path = 'data/models/face_model_eigen.yml'
faceRecognizerEigen = cv2.face.EigenFaceRecognizer_create(num_components=50,threshold=4000.0)
faceRecognizerEigen.train(imagesFaceTrain,np.array(labelsFaceTrain))
faceRecognizerEigen.write(save_path)

Trainiere mit Eigen-Faces


In [ ]:
print("Trainiere mit Fisher-Faces")
save_path = 'data/models/face_model_fisher.yml'
faceRecognizerFisher = cv2.face.FisherFaceRecognizer_create(num_components=50,threshold=4000.0)
faceRecognizerFisher.train(imagesFaceTrain,np.array(labelsFaceTrain))
faceRecognizerFisher.write(save_path)

Trainiere mit Fisher-Faces


In [ ]:
print("Trainiere mit LBP-Histogrammen")
save_path = 'data/models/face_model_lbph.yml'
faceRecognizerLBPH = cv2.face.LBPHFaceRecognizer_create()
faceRecognizerFisher.train(imagesFaceTrain,np.array(labelsFaceTrain))
faceRecognizerLBPH.write(save_path)
# Label-Nummer auf Label-Namen mappen
with open('data/models/labels_map.pkl', 'wb') as f:
    cPickle.dump(labelsMap, f)

Trainiere mit LBP-Histogrammen


In [20]:
faceRecognizerFisher.write('data/models/face_model_fisher.yml')
faceRecognizerEigen.write('data/models/face_model_eigen.yml')
faceRecognizerLBPH.write('data/models/face_model_lbph.yml')